In [1]:
import numpy as np
from numpy.linalg import eigh
from scipy.sparse import diags
from scipy.linalg import eigvalsh
import cvxpy as cp

In [3]:
def gen_CQKFWP(n, seed=0, rank=1.1, conv=1, ecc=0.99, dom=1, box=True):
    if not isinstance(n, int) or n <= 0:
        raise ValueError("n must be a positive integer")

    np.random.seed(seed)

    # Generate Q
    r = int(round(rank * n))
    p = int(round(r * conv))
    Q = np.zeros((n, n))

    if p > 0:
        G = np.random.rand(p, n)
        Q += G.T @ G
    if r > p:
        G = np.random.rand(r - p, n)
        Q -= G.T @ G

    # Modify the diagonal if dom != 1
    if dom != 1:
        diag_indices = np.diag_indices_from(Q)
        diag_vals = np.diag(Q)
        D = diags(diag_vals * (1 + (2 * np.random.rand(n) - 1) / 3) * dom).toarray()
        Q[diag_indices] = D[diag_indices]

    Q = (Q + Q.T) / 2  # Ensure symmetry
    vals, V = eigh(Q)

    # Compute eigenvalue decomposition and modify eccentricity
    if vals[0] > 1e-14:
        lambd_n = vals[0]
        lambd_1 = vals[-1]
        l = lambd_n * np.ones(n) + ((lambd_n / (lambd_1 - lambd_n))) * (2 * ecc / (1 - ecc)) * (vals - lambd_n)
        Q = V @ np.diag(l) @ V.T

    # Generate q
    x = 2 * abs(box) * np.random.rand(n) - abs(box)
    q = -Q @ x

    # Randomly modify q if box < 0 and Q is rank-deficient
    if box and vals[0] <= 1e-14:
        q *= (4 / 3) * np.random.rand(n) - 2 / 3

    # Compute v
    if vals[0] > 1e-14 or (vals[0] > -1e-14 and box > 0):
        v = np.dot(q, x) / 2
    else:
        v = -np.inf

    # Generate a, b, l, u
    a = np.random.rand(n)
    while np.linalg.norm(a) == 0:
        a = np.random.rand(n)

    b = np.random.rand()

    l = np.random.rand(n)
    u = l + np.random.rand(n)

    QP = {
        "Q": Q,
        "q": q,
        "v": v,
        "a": a,
        "b": b,
        "l": l,
        "u": u
    }
    return QP


In [4]:
def linear_subproblem(c, a, b, l, u):
    n = len(c)
    x_star = []
    for i in range(n):
        if c[i] >= 0:
            x_star.append(l[i])
        else:
            x_star.append(u[i])
    
    if (a @ x_star) >= b:
        print("In the box")
        return x_star
    
    # The minimum value of the objective function is not in feasible region
    print("Not in the vertex of the hyperbox")
    x_pack = [(i, x_star[i], c[i], a[i]) for i in range(n)]
    # Order in ascending order of c_i/a_i
    x_pack = sorted(x_pack, key=lambda x: x[2] / x[3], reverse=False)
    ordered_index = [t[0] for t in x_pack]
    
    for idx in ordered_index:
        x_star[idx] = u[idx]
        
        if (a @ x_star) > b:
            x_star[idx] = (b - (a @ x_star) + (a[idx] * x_star[idx])) / a[idx]
            return x_star
        
        
    print("No solution found")
    return x_star

In [5]:
def CQKFWP(BCQP, t=0, eps=1e-6, MaxIter=1000, verbose=True):
    """
    Apply the (possibly, stabilized) Franke-Wolfe algorithm with exact line
    search to the convex Box-Constrained Quadratic program
    
    (P) min { (1/2) x^T * Q * x + q * x : ax >= b, l <= x <= u }
    
    Parameters:
    - BCQP: A dictionary encoding the Convex Quadratic Knapsack to be solved.
    - t: Stabilization parameter. If t > 0, the new point is chosen in the
         box of relative size t around the current point.
    - eps: The accuracy in the stopping criterion.
    - MaxIter: The maximum number of iterations.
    
    Returns:
    - v: The best function value found so far.
    - x (optional): The best solution found so far.
    - status (optional): A string describing the status of the algorithm at termination.
    """
    # Initializations
    n = BCQP['q'].shape[0]
    x = BCQP['u'] / 2  # Start from the middle of the box
    best_lower_bound = -np.inf  # Best lower bound so far
    Verbose = verbose

    if Verbose:
        print("Frank-Wolfe method")
        print("iter\tf(x)\t\tlower_bound\t\tgap\n")

    i = 1

    while True:
        # Compute function value and direction = - gradient
        v = 0.5 * np.dot(x.T, np.dot(BCQP['Q'], x)) + np.dot(BCQP['q'].T, x)
        g = np.dot(BCQP['Q'], x) + BCQP['q']

        # Solve min { <g, z> : az >= b, l <= z <= u }
        y = linear_subproblem(g, BCQP['a'], BCQP['b'], BCQP['l'], BCQP['u'])

        # Compute the lower bound
        lower_bound = v + np.dot(g.T, y - x)
        if lower_bound > best_lower_bound:
            best_lower_bound = lower_bound

        # Compute the relative gap
        gap = (v - best_lower_bound) / max(abs(v), 1)

        if Verbose:
            print(f"{i}\t{v:.8e}\t{best_lower_bound:.8e}\t{gap:.4e}")

        # Stopping criteria
        if gap <= eps:
            status = 'optimal'
            break
        if i > MaxIter:
            status = 'stopped'
            break

        # In the stabilized case, restrict y in the box
        if t > 0:
            y = np.maximum(x - t * BCQP['u'], np.minimum(x + t * BCQP['u'], y))

        # Compute step size
        d = y - x
        den = np.dot(d.T, np.dot(BCQP['Q'], d))
        if den <= 1e-16:
            alpha = 1  # F is linear along d, take max possible stepsize
        else:
            alpha = min(-np.dot(g.T, d) / den, 1)

        # Compute new point
        x = x + alpha * d
        i += 1

    if Verbose:
        print(f"stop: {i} iter, status = {status}, fbest = {v:.8e}, gap = {gap:.4e}")

    return v, x, status


In [40]:
def basic_CQKFWP(BCQP, eps=1e-6, MaxIter=1000, verbose=True):
    """
    Apply the basic Franke-Wolfe algorithm with exact line
    search to the convex Box-Constrained Quadratic program
    
    (P) min { (1/2) x^T * Q * x + q * x : ax >= b, l <= x <= u }
    """
    
    x = BCQP['u'] / 2  # Start from the middle of the box

    if verbose:
        print("Frank-Wolfe method")

    for i in range(MaxIter):
        v = 0.5 * np.dot(x.T, np.dot(BCQP['Q'], x)) + np.dot(BCQP['q'].T, x)

        if verbose:
            print(f"{i}: f(x):{v:.4e}")

        c = np.dot(BCQP['Q'], x) + BCQP['q']

        x_bar = linear_subproblem(c, BCQP['a'], BCQP['b'], BCQP['l'], BCQP['u'])
        d = x_bar - x
        
        if np.dot(c.T, d) <= eps:
            break
        
        den = np.dot(d.T, np.dot(BCQP['Q'], d))
        if den <= 1e-16:
            alpha = 1  # F is linear along d, take max possible stepsize
        else:
            alpha = min(-np.dot(c.T, d) / den, 1)
        
        x = x + alpha * d
    
    return v, x

In [41]:
problem = gen_quadratic_knapsack_problem(10, seed=1, rank=1.1, conv=1, ecc=0.99, dom=1, box=True)
v1, x1, _ = FWBCQP(problem, t=0, eps=1e-6, MaxIter=1000, verbose=True)
basic_KS_FWBCQ(problem, eps=1e-6, MaxIter=1000, verbose=True)

Frank-Wolfe method
iter	f(x)		lower_bound		gap

In the box
1	2.78012018e+00	5.95848542e+00	-1.1432e+00
stop: 1 iter, status = optimal, fbest = 2.78012018e+00, gap = -1.1432e+00
Frank-Wolfe method
0: f(x):2.7801e+00
In the box
1: f(x):-2.0855e-01
In the box


(-0.20854552078228236,
 array([-0.23038936, -0.16072986,  0.16656216, -0.28836212,  0.05224329,
        -0.0302882 ,  0.85614799,  0.42593822, -0.31088457, -0.14125547]))

In [35]:
# (P) min { (1/2) x^T * Q * x + q * x : ax >= b, l <= x <= u }

# Define variables
Q = problem['Q']    
q = problem['q']
a = problem['a']
b = problem['b']
l = problem['l']
u = problem['u']
n = len(q)

x = cp.Variable(n)

# Define constraints
constraints = [a @ x >= b, x >= l, x <= u]

# Define objective
objective = cp.Minimize(0.5 * cp.quad_form(x, Q) + q.T @ x)

# Define problem
P = cp.Problem(objective, constraints)
P.solve()

# Print result
print(f"Optimal value: {P.value}")
print(f"Optimal x: {x.value}")

Optimal value: 6.803511818676445
Optimal x: [0.96959575 0.56103022 0.01864729 0.80063267 0.23297427 0.8071052
 0.38786064 0.86354185 0.74712164 0.55624023]


In [39]:
tries = 50
failed = 0

for i in range(11, tries+11):
    
    data = gen_quadratic_knapsack_problem(10, seed=i, rank=1.1, conv=1, ecc=0.99, dom=1, box=True)

    # Define variables
    Q = data['Q']
    q = data['q']
    a = data['a']
    b = data['b']
    l = data['l']
    u = data['u']
    n = len(q)

    x = cp.Variable(n)
    # Define constraints
    constraints = [a @ x >= b, x >= l, x <= u]

    # Define objective
    objective = cp.Minimize(0.5 * cp.quad_form(x, Q) + q.T @ x)

    # Define problem
    P = cp.Problem(objective, constraints)
    P.solve()
    gold_solution = [P.value, x.value]
    y, x_opt = basic_KS_FWBCQ(data, eps=1e-6, MaxIter=1000, verbose=False)
    our_solution = [y, x_opt]

    if not np.allclose(gold_solution[1], our_solution[1]):
        print(f"Test {i} failed")
        failed += 1

if failed == 0:
    print(f"All {tries} tests passed")
else:
    print(f"{failed} out of {tries} tests failed")
    

In the box
Test 11 failed
In the box
Test 12 failed
In the box
In the box
In the box
Test 14 failed
In the box
In the box
In the box
In the box
In the box
In the box
In the box
Test 18 failed
In the box
Test 19 failed
In the box
Test 20 failed
In the box
In the box
In the box
In the box
In the box
Test 23 failed
In the box
In the box
In the box
In the box
In the box
Test 26 failed
In the box
Test 27 failed
In the box
In the box
In the box
Test 29 failed
In the box
In the box
In the box
Test 31 failed
In the box
In the box
In the box
In the box
In the box
Test 34 failed
In the box
Test 35 failed
In the box
Test 36 failed
In the box
Test 37 failed
In the box
In the box
In the box
Test 39 failed
In the box
In the box
In the box
Test 41 failed
In the box
In the box
In the box
In the box
In the box
Test 44 failed
In the box
Test 45 failed
In the box
Test 46 failed
In the box
In the box
In the box
Test 48 failed
In the box
In the box
In the box
Test 50 failed
In the box
Test 51 failed
In the

PROBLEMI:
non genera problemi che non sono con il minimo degli angoli

frank wolfe non funziona